# 🚗🤖 [실습] CNN(합성곱 신경망) 계산과정 완전 정복하기: 6×6 픽셀부터 손글씨(1 vs 2) 판별까지

> ### 🗺️ 이 모듈에서 배우는 것 (3줄 요약)
> 1. 컴퓨터가 6×6 픽셀 그림을 숫자 행렬로 읽는 방법을 알아봐요.
> 2. 작은 필터(커널: 모양 찾기 돋보기 🔍)로 선·모양을 찾아내는 **합성곱**을 직접 계산해 봐요.
> 3. 찾은 특징을 꾹꾹 눌러 압축(풀링·평탄화)한 뒤, 숫자 1과 2를 판별하는 AI의 전체 과정을 완성해요!
>
> ### 🎒 준비물 & 선수 지식
> - **준비물**: 구글 코랩(Colab) + 엑셀 실습 파일(`CNN_계산과정_숫자판별_정리본.xlsx`) + 웹 시뮬레이터(`cnn_classifier_simulator.html`)를 함께 열어 두세요.
> - **선수 지식**: 파이썬을 처음 봐도 괜찮아요! 사칙연산 정도면 충분하고, 엑셀에서 수식을 써본 경험이 있다면 더 좋아요. 😊

안녕하세요! 이 노트북은 **엑셀(Excel) 실습 파일(`CNN_계산과정_숫자판별_정리본.xlsx`)**과 **웹 시뮬레이터(`cnn_classifier_simulator.html`)**에서 다룬 
**CNN(합성곱 신경망)의 전체 수학 연산 과정**을 파이썬 코드로 직접 한 단계씩 실행해보는 실습 자료입니다.

---

### 📚 실습 목표
1. **이미지 행렬 이해**: 컴퓨터가 $6 \times 6$ 흑백 이미지(0~255)를 어떻게 2차원 행렬로 다루는지 이해합니다.
2. **데이터 정규화(Normalization)**: $0 \sim 255$ 범위의 정수를 $0.0 \sim 1.0$ 사이 소수로 변환하는 이유를 배웁니다.
3. **합성곱(Convolution)**(작은 필터를 그림 위로 한 칸씩 밀어가며 겹치는 부분을 곱한 뒤 모두 더하는 연산): 세로선, 가로선, 대각선 필터(커널: 특정 모양을 찾는 작은 3×3 숫자판)를 사용해 이미지의 고유 특징을 추출하는 원리를 이해합니다.
4. **맥스풀링(Max Pooling)**(특성맵을 작은 구역으로 나누어 각 구역의 가장 큰 값 하나만 남기는 압축): 특징맵의 크기를 절반으로 줄이며 가장 강한 특징을 유지하는 차원 축소를 실습합니다.
5. **완전연결층(Dense Layer)**(각 특징에 가중치를 곱하고 편향을 더해 최종 점수를 만드는 마지막 계산 단계): 추출된 8차원 특징 벡터와 가중치($W$), 편향($b$)의 내적을 통해 최종적으로 **숫자 1인지 2인지** 판별합니다.


---
## 1. 실습 환경 준비 (라이브러리 불러오기)

파이썬에서 행렬 연산을 지원하는 **NumPy**와 시각화를 위한 **Matplotlib**을 불러옵니다.  
코랩 환경에서 한글 깨짐을 방지하는 폰트 설정 코드도 포함되어 있습니다.


In [ ]:
import numpy as np

# 시각화 라이브러리 (Colab 기본 지원)
try:
    import matplotlib.pyplot as plt
    # 한글 폰트 설정 (Mac/Colab 호환)
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['font.family'] = 'sans-serif'
    HAS_MATPLOTLIB = True
    print("✅ NumPy와 Matplotlib이 성공적으로 준비되었습니다!")
except ImportError:
    HAS_MATPLOTLIB = False
    print("⚠️ Matplotlib이 설치되어 있지 않아 텍스트/표로 시각화합니다.")


---
## 2. [1단계] 6×6 픽셀 손글씨 이미지 생성 (0 ~ 255)

우리가 손으로 쓴 숫자 '1'과 숫자 '2'를 가로 6칸, 세로 6칸 총 36개의 작은 사각형(픽셀)으로 표현해 봅시다.
* 검은 바탕: `0`
* 흰색 글씨 획: `255`
* (명암이 있는 회색 획: `1~254`)

> ✅ **체크포인트**: 아래 코드 셀을 실행하면 숫자 1·2의 6×6 행렬이 출력돼요. 흰 글씨는 255, 검은 바탕은 0으로 찍히는지 눈으로 확인해 보세요!


In [ ]:
# 숫자 1 이미지 행렬 (세로선 2줄이 강조됨)
img_digit1 = np.array([
    [  0,   0, 255, 255,   0,   0],
    [  0,   0, 255, 255,   0,   0],
    [  0,   0, 255, 255,   0,   0],
    [  0,   0, 255, 255,   0,   0],
    [  0,   0, 255, 255,   0,   0],
    [  0,   0, 255, 255,   0,   0]
], dtype=float)

# 숫자 2 이미지 행렬 (가로선과 대각선, 꺾임이 강조됨)
img_digit2 = np.array([
    [  0, 255, 255, 255, 255,   0],
    [  0,   0,   0,   0, 255,   0],
    [  0,   0, 255, 255,   0,   0],
    [  0, 255,   0,   0,   0,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0, 255, 255, 255, 255,   0]
], dtype=float)

print("📌 숫자 1 행렬 (6x6):")
print(img_digit1)
print("\n📌 숫자 2 행렬 (6x6):")
print(img_digit2)


### 🖼️ 이미지 시각화 함수 정의
숫자로 된 2차원 행렬을 시각적인 격자 그림으로 확인해 봅시다.


In [ ]:
def plot_matrix(matrix, title, cmap='Blues', fmt='.0f', vmin=None, vmax=None):
    if not HAS_MATPLOTLIB:
        print(f"=== {title} ===")
        print(matrix)
        return
    
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    
    # 격자 선 및 틱 설정
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_xticklabels(np.arange(matrix.shape[1]))
    ax.set_yticklabels(np.arange(matrix.shape[0]))
    
    # 각 셀 안에 숫자 표기
    threshold = (matrix.max() + matrix.min()) / 2.0 if matrix.max() != matrix.min() else 0.5
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = "white" if val > threshold else "black"
            text_str = f"{val:{fmt}}" if abs(val) >= 0.001 else "0"
            ax.text(j, i, text_str, ha="center", va="center", color=color, fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# 숫자 1과 2 시각화
plot_matrix(img_digit1, "숫자 1 원본 이미지 (6x6 픽셀, 0~255)", cmap='gray_r', vmin=0, vmax=255)
plot_matrix(img_digit2, "숫자 2 원본 이미지 (6x6 픽셀, 0~255)", cmap='gray_r', vmin=0, vmax=255)


---
## 3. [2단계] 데이터 정규화 (Normalization)

컴퓨터가 $0 \sim 255$처럼 큰 숫자를 그대로 곱하고 더하면 계산 도중 숫자가 너무 커져 컴퓨터 메모리가 넘치거나(오버플로우), 
인공지능 학습이 불안정해집니다. 따라서 모든 픽셀 값을 255로 나누어 **$0.0 \sim 1.0$ 사이의 소수**로 변환합니다.

$$\text{정규화 픽셀} = \frac{\text{원본 픽셀}}{255.0}$$

> **엑셀 실습과의 연계:** 엑셀 시트 2열의 `=A5/255` 수식과 동일합니다!

> ✅ **체크포인트**: 코드를 실행하고 모든 값이 0.0~1.0 사이의 소수로 바뀌었는지 확인해 보세요. 가장 밝은 글씨는 1.0이 되어야 해요!


In [ ]:
# 정규화 연산
norm1 = img_digit1 / 255.0
norm2 = img_digit2 / 255.0

print("📌 숫자 1 정규화 결과 (0.0 ~ 1.0):")
print(np.round(norm1, 2))

plot_matrix(norm1, "숫자 1 정규화 행렬 (0.0 ~ 1.0)", cmap='Purples', fmt='.1f', vmin=0, vmax=1)


---
## 4. [3단계] 4종 특징 추출 필터 (3×3 커널) 준비

CNN의 핵심은 이미지 전체를 훑으며 **특정 모양(세로선, 가로선, 대각선 등)**이 있는지 감지하는 **필터(커널)**입니다.  
우리는 4가지 필터를 사용합니다:
1. **세로선 필터 (|)**: $\begin{bmatrix}0&1&0\\0&1&0\\0&1&0\end{bmatrix}$ (가운데 세로 획 탐지)
2. **가로선 필터 (-)**: $\begin{bmatrix}0&0&0\\1&1&1\\0&0&0\end{bmatrix}$ (가운데 가로 획 탐지)
3. **대각선 (/) 필터**: $\begin{bmatrix}0&0&1\\0&1&0\\1&0&0\end{bmatrix}$ (우상향 대각 획 탐지)
4. **역대각선 (\) 필터**: $\begin{bmatrix}1&0&0\\0&1&0\\0&0&1\end{bmatrix}$ (좌상향 대각 획 탐지)

> ✅ **체크포인트**: 4개의 3×3 필터가 출력되면 성공! 각 필터가 어떤 모양(세로선·가로선·대각선)을 찾는 돋보기인지 친구에게 설명해 보세요. 🔍


In [ ]:
# 4개 커널 정의
kernel_v = np.array([
    [0, 1, 0],
    [0, 1, 0],
    [0, 1, 0]
], dtype=float)

kernel_h = np.array([
    [0, 0, 0],
    [1, 1, 1],
    [0, 0, 0]
], dtype=float)

kernel_d1 = np.array([
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0]
], dtype=float)

kernel_d2 = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=float)

kernels = [kernel_v, kernel_h, kernel_d1, kernel_d2]
kernel_names = ["1. 세로선 필터 (|)", "2. 가로선 필터 (-)", "3. 대각선 (/) 필터", "4. 역대각선 (\\) 필터"]

for name, k in zip(kernel_names, kernels):
    print(f"=== {name} ===")
    print(k)


---
## 5. [4단계] 2차원 합성곱 연산 (2D Convolution)

합성곱 연산은 $3 \times 3$ 필터를 $6 \times 6$ 이미지의 왼쪽 위부터 오른쪽 아래까지 1칸씩(Stride=1) 이동시키며,  
**필터와 겹치는 $3 \times 3$ 영역의 픽셀들을 원소별로 곱한 뒤 모두 더하는 연산**입니다.

$$O(i, j) = \sum_{m=0}^{2} \sum_{n=0}^{2} I(i+m, j+n) \times K(m, n)$$

* 입력 크기: $6 \times 6$
* 필터 크기: $3 \times 3$
* 출력(특성맵) 크기: $(6 - 3) + 1 = 4 \times 4$

> **엑셀 실습과의 연계:** 엑셀의 `=SUMPRODUCT(I5:K7, O$5:Q$7)` 수식과 동일합니다!

> ✅ **체크포인트**: 숫자 1의 세로선(|) 특성맵 최대값이 3.0으로 가장 크게 나오나요? 숫자 1은 세로선의 제왕이니까요! 👑


In [ ]:
def conv2d(image, kernel):
    """2차원 순수 합성곱 연산 (Stride=1, Padding=0)"""
    in_h, in_w = image.shape
    k_h, k_w = kernel.shape
    # 출력 특성맵 크기 = 입력 크기 - 필터 크기 + 1  (6 - 3 + 1 = 4)
    out_h = in_h - k_h + 1
    out_w = in_w - k_w + 1
    
    feature_map = np.zeros((out_h, out_w), dtype=float)
    
    # 필터를 이미지의 왼쪽 위부터 오른쪽 아래까지 1칸씩 이동시키며
    for i in range(out_h):
        for j in range(out_w):
            # 3x3 수용장(Receptive Field) 추출
            window = image[i:i+k_h, j:j+k_w]
            # 원소별 곱셈 후 총합
            feature_map[i, j] = np.sum(window * kernel)
            
    return np.round(feature_map, 2)

# 숫자 1에 대해 4개 필터 적용
fmaps1 = [conv2d(norm1, k) for k in kernels]

print("✅ 숫자 1 - 4개 특성맵 (4x4):")
for name, fmap in zip(kernel_names, fmaps1):
    print(f"\n[{name}] 특성맵 (최대값: {fmap.max()}):")
    print(fmap)


### 🔍 특성맵 비교 분석: 숫자 1 vs 숫자 2
두 숫자가 4가지 필터를 거쳤을 때 어떤 특성맵이 나오는지 시각화로 비교해 봅시다!

> ✅ **체크포인트**: 숫자 2에서는 가로선(-) 필터의 최대값이 3.0까지 올라가는지 확인해 보세요. 숫자 1과 무엇이 다른가요?


In [ ]:
# 숫자 2에 대해서도 4개 필터 적용
fmaps2 = [conv2d(norm2, k) for k in kernels]

print("📊 [숫자 1] 필터별 특성맵 최대 반응 강도:")
for name, fmap in zip(kernel_names, fmaps1):
    print(f"  • {name:20s}: 최대값 {fmap.max():.2f}")

print("\n📊 [숫자 2] 필터별 특성맵 최대 반응 강도:")
for name, fmap in zip(kernel_names, fmaps2):
    print(f"  • {name:20s}: 최대값 {fmap.max():.2f}")

# 특성맵 시각화
plot_matrix(fmaps1[0], "숫자 1 - 세로선(|) 특성맵 (강한 반응: 3.0)", cmap='Greens', fmt='.1f')
plot_matrix(fmaps2[1], "숫자 2 - 가로선(-) 특성맵 (강한 반응: 3.0)", cmap='Blues', fmt='.1f')


---
## 6. [5단계] 맥스풀링 (Max Pooling 2×2, Stride=2)

합성곱으로 얻은 $4 \times 4$ 특성맵을 $2 \times 2$ 크기의 4개 구역으로 나누고, **각 구역에서 가장 큰 값(최대값) 하나만 선택**합니다.
* 크기 변화: $4 \times 4 \rightarrow 2 \times 2$ (데이터 크기 75% 절약)
* 장점: 사소한 위치 변화나 노이즈가 있어도 가장 두드러진 특징을 보존합니다(위치 불변성).

> **엑셀 실습과의 연계:** 엑셀의 `=MAX(S5:T6)` 수식과 동일합니다!

> ✅ **체크포인트**: 4×4 특성맵이 2×2 풀링맵으로 줄었나요? 데이터가 16개 → 4개, 즉 75%나 줄었는지 확인해 보세요!


In [ ]:
def max_pool2d(feature_map, pool_size=2):
    """2x2 맥스풀링 연산 (Stride=pool_size)"""
    in_h, in_w = feature_map.shape
    out_h = in_h // pool_size
    out_w = in_w // pool_size
    
    pooled = np.zeros((out_h, out_w), dtype=float)
    
    for r in range(out_h):
        for c in range(out_w):
            window = feature_map[r*pool_size:(r+1)*pool_size, c*pool_size:(c+1)*pool_size]
            # 2x2 구역 안에서 가장 큰 값 1개만 쏙 뽑아내기
            pooled[r, c] = np.max(window)
            
    return pooled

pools1 = [max_pool2d(f) for f in fmaps1]
pools2 = [max_pool2d(f) for f in fmaps2]

print("✅ 숫자 1의 4개 풀링맵 (2x2):")
for name, p in zip(kernel_names, pools1):
    print(f"\n[{name}] 풀링맵:\n{p}")


---
## 7. [6단계] 평탄화(Flattening): 2D 풀링맵 → 1D 특성 벡터 변환

2차원 풀링맵들에서 주요 특징들을 뽑아내어 일렬로 길게 늘어선 **1차원 벡터($X$)**로 만듭니다.  
엑셀 실습 교재에서는 4개 풀링맵의 첫 번째 행(상단 2개 원소)을 순서대로 추출하여 **총 8차원 특징 벡터**를 구성했습니다:

$$X = [x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8]$$
* $x_1, x_2$: 세로선 풀링맵 상단 2개
* $x_3, x_4$: 가로선 풀링맵 상단 2개
* $x_5, x_6$: 대각선(/) 풀링맵 상단 2개
* $x_7, x_8$: 역대각선(\\) 풀링맵 상단 2개

> ✅ **체크포인트**: 8개의 숫자로 이루어진 1줄짜리 특성 벡터가 완성됐나요? (x1부터 x8까지!)


In [ ]:
def extract_vector(pools):
    # 각 2x2 풀링맵의 첫 번째 행(상단 2개 값)만 순서대로 꺼내 1줄로 길게 연결해요
    vec = []
    for p in pools:
        vec.append(p[0, 0])
        vec.append(p[0, 1])
    return np.array(vec, dtype=float)

vec1 = extract_vector(pools1)
vec2 = extract_vector(pools2)

feature_labels = [
    "x1(세로선1)", "x2(세로선2)",
    "x3(가로선1)", "x4(가로선2)",
    "x5(대각선/1)", "x6(대각선/2)",
    "x7(역대각선1)", "x8(역대각선2)"
]

print("📌 숫자 1의 8차원 특성 벡터 (X1):")
for label, val in zip(feature_labels, vec1):
    print(f"  • {label:12s} = {val:.2f}")

print("\n📌 숫자 2의 8차원 특성 벡터 (X2):")
for label, val in zip(feature_labels, vec2):
    print(f"  • {label:12s} = {val:.2f}")


---
## 8. [7단계] 완전연결층(Dense Layer): 가중치($W$)와 편향($b$) 내적 연산

최종 판별을 위해 각 특징의 중요도를 결정하는 **가중치($W$)**를 곱하고 기본 기준점인 **편향($b$)**을 더합니다.

$$\text{Score} = X \cdot W + b = \sum_{i=1}^{8} (x_i \times w_i) + b$$

* **세로선 가중치 ($w_1, w_2 = +3, +3$)**: 숫자 1의 상징이므로 **양수(+3)**를 곱해 점수를 높여줍니다.
* **가로선 & 대각선 가중치 ($w_3 \sim w_6 = -3, -3, -3, -3$)**: 숫자 2의 상징이므로 **음수(-3)**를 곱해 점수를 깎아줍니다.
* **편향 ($b = +5.0$)**: 기본 기준점을 제공합니다.

> **엑셀 실습과의 연계:** 엑셀의 `=SUMPRODUCT(AB5:AB12, AC5:AC12) + AD5` 수식과 100% 동일합니다!

> ✅ **체크포인트**: 숫자 1의 최종 점수는 양수(+3.00), 숫자 2의 최종 점수는 음수(-3.00)가 나와야 정상이에요!


In [ ]:
# 모델 가중치 및 편향
weights = np.array([3.0, 3.0, -3.0, -3.0, -3.0, -3.0, 1.0, 1.0])
bias = 5.0

def calculate_dense_layer(x_vec, digit_name):
    element_products = x_vec * weights
    sum_xw = np.sum(element_products)
    final_score = sum_xw + bias
    
    print(f"==================================================")
    print(f"        🎯 [{digit_name}] 완전연결층 계산 과정 상세")
    print(f"==================================================")
    print(f"{'번호':<4} | {'특징명':<10} | {'입력(X)':<7} | {'가중치(W)':<8} | {'가중치 곱(X*W)':<12}")
    print("-" * 52)
    for i in range(8):
        print(f"x{i+1:<3} | {feature_labels[i]:<10} | {x_vec[i]:>6.2f}  | {weights[i]:>7.1f}  | {element_products[i]:>11.2f}")
    print("-" * 52)
    print(f"▶ 가중치 곱 총합 (∑ X*W) = {sum_xw:>+.2f}")
    print(f"▶ 편향 (b)              = {bias:>+.2f}")
    print(f"▶ 최종 점수 (Score)      = {final_score:>+.2f}")
    
    # 판별 규칙: Score >= 0 이면 1, Score < 0 이면 2
    predicted = 1 if final_score >= 0 else 2
    print(f"▶ 최종 예측 결과         = [ 숫자 {predicted} ]")
    print(f"==================================================\n")
    return final_score, predicted

score1, pred1 = calculate_dense_layer(vec1, "숫자 1")
score2, pred2 = calculate_dense_layer(vec2, "숫자 2")


---
## 9. [8단계] 확률 변환 (시그모이드 함수)

실제 인공지능 모델에서는 최종 점수(Logit)를 **0% ~ 100% 사이의 확률**로 바꾸기 위해 **시그모이드(Sigmoid) 함수**나 **소프트맥스(Softmax) 함수**를 사용합니다:

$$P(Y=1) = \sigma(\text{Score}) = \frac{1}{1 + e^{-\text{Score}}}, \quad P(Y=2) = 1 - P(Y=1)$$

> ✅ **체크포인트**: '숫자 1일 확률'과 '숫자 2일 확률'을 더하면 100%가 되나요? 직접 더해 보세요! ➕


In [ ]:
def sigmoid(z):
    # 오버플로우 방지 클리핑
    z_clip = np.clip(z, -15, 15)
    return 1.0 / (1.0 + np.exp(-z_clip))

prob1_digit1 = sigmoid(score1)
prob2_digit1 = 1.0 - prob1_digit1

prob1_digit2 = sigmoid(score2)
prob2_digit2 = 1.0 - prob1_digit2

print(f"📊 [숫자 1 이미지 판별 확률]")
print(f"  • 숫자 1일 확률: {prob1_digit1 * 100:>6.2f} %")
print(f"  • 숫자 2일 확률: {prob2_digit1 * 100:>6.2f} %")

print(f"\n📊 [숫자 2 이미지 판별 확률]")
print(f"  • 숫자 1일 확률: {prob1_digit2 * 100:>6.2f} %")
print(f"  • 숫자 2일 확률: {prob2_digit2 * 100:>6.2f} %")


---
## 10. [한눈에 보는 통합 파이프라인] 전체 CNN 분류 함수 정의

지금까지 작성한 모든 단계를 단 하나의 함수 `predict_digit_cnn(img_6x6)`로 묶어봅시다!  
어떤 $6 \times 6$ 이미지가 들어와도 곧바로 판별 결과를 출력해 줍니다.

> ✅ **체크포인트**: 어떤 6×6 이미지를 넣어도 바로 판별 결과가 나오면 성공! 이제 여러분도 CNN 설계자예요. 🎉


In [ ]:
def predict_digit_cnn(img_6x6, show_details=True):
    """6x6 이미지를 입력받아 CNN 전방향 연산을 수행하고 1 또는 2를 판별하는 함수"""
    # 1. 정규화
    norm = img_6x6 / 255.0
    
    # 2. 합성곱 (4개 특성맵)
    fmaps = [conv2d(norm, k) for k in kernels]
    
    # 3. 맥스풀링 (4개 풀링맵)
    pools = [max_pool2d(f) for f in fmaps]
    
    # 4. 벡터 추출 (8차원)
    vec = extract_vector(pools)
    
    # 5. 완전연결층 연산
    score = np.dot(vec, weights) + bias
    
    # 6. 예측 및 확률
    pred = 1 if score >= 0 else 2
    prob1 = sigmoid(score)
    prob2 = 1.0 - prob1
    
    if show_details:
        print(f"--------------------------------------------------")
        print(f"🤖 CNN 판별 결과: [ 숫자 {pred} ] (최종 점수: {score:+.2f})")
        print(f"📊 확신도: 숫자 1 ({prob1*100:.1f}%) vs 숫자 2 ({prob2*100:.1f}%)")
        print(f"--------------------------------------------------")
        
    return pred, score, prob1

# 테스트 실행
pred, score, prob1 = predict_digit_cnn(img_digit1)
pred, score, prob1 = predict_digit_cnn(img_digit2)


---
## 11. 🧪 [학생 자율 탐구 활동] 나만의 숫자를 그려서 테스트해보기!

아래 격자의 0을 255로 바꾸어 **나만의 손글씨**를 직접 디자인하고, 인공지능이 어떻게 판별하는지 실험해 보세요!

### 💡 탐구 미션:
1. **미션 1**: 숫자 1의 세로선을 오른쪽으로 1칸 또는 2칸 옮겨서 그려보세요. 여전히 1로 잘 맞추나요?
2. **미션 2**: 완벽한 대각선(/) 하나만 그렸을 때 인공지능은 1과 2 중 무엇으로 판별할까요?
3. **미션 3**: 십자가(+)나 네모(□)를 그렸을 때 왜 엉뚱한 결과가 나오는지 설명해 보세요.


In [ ]:
# 👇 아래 배열의 숫자를 자유롭게 수정해 보세요! (0: 검은 바탕, 255: 글씨)
my_custom_digit = np.array([
    [  0,   0,   0, 255, 255,   0],
    [  0,   0,   0, 255, 255,   0],
    [  0,   0,   0, 255, 255,   0],
    [  0,   0,   0, 255, 255,   0],
    [  0,   0,   0, 255, 255,   0],
    [  0,   0,   0, 255, 255,   0]
], dtype=float)

# 시각화 및 판별
plot_matrix(my_custom_digit, "내가 그린 손글씨 (6x6)", cmap='gray_r', vmin=0, vmax=255)
predict_digit_cnn(my_custom_digit)


---
## 12. 🎓 수업 정리 및 토론 과제

오늘 실습을 통해 우리는 현대 인공지능이 이미지를 인식하는 가장 핵심적인 원리인 **합성곱 신경망(CNN)**의 전 과정을 살펴보았습니다.

### 📝 핵심 정리 질문:
1. **Q1. 일반 인공신경망 대신 CNN을 사용하는 가장 결정적인 이유는 무엇인가요?**  
   * **A**: 일반 신경망은 2차원 이미지를 1차원으로 펴버려 공간적 형태(선, 모서리 등)가 파괴되지만, CNN은 작은 필터를 통해 공간적 특징을 그대로 추출하기 때문입니다.

2. **Q2. 맥스풀링(Max Pooling)의 두 가지 핵심 역할은 무엇인가요?**  
   * **A**: 계산량을 대폭 줄여주고(차원 축소), 사물의 위치가 조금 이동하거나 틀어져도 주요 특징을 감지할 수 있게 해줍니다(위치 불변성).

3. **Q3. 엑셀 실습과 파이썬 코딩 실습의 차이점과 공통점은 무엇인가요?**  
   * **공통점**: 똑같은 수학 수식(`SUMPRODUCT`, `MAX`, `IF`)을 기반으로 동작합니다.
   * **차이점**: 엑셀은 모든 중간 계산 과정을 시각적으로 펼쳐놓고 관찰하기에 좋고, 파이썬은 반복문과 함수로 수많은 이미지를 초고속으로 자동 처리할 수 있습니다.
